# Module 3 — Robust Evaluation

**By the end of this notebook, you will be able to:**
- Explain why a single manual train/test pair gives a fragile, direction-dependent performance estimate
- Use `GroupKFold` to evaluate a model against every group in turn, without ever mixing a group across train and validation
- Read a mean **and** a std across folds, not just a single number

**Context:** In Session 2's reflection, you compared training on Kampala and testing on Nairobi, then reversed it — and got two different, both disappointing results. That comparison only used two of the four cities, and only one direction each. This module systematizes it: evaluate against every city, one at a time, on the full 4-city dataset you cleaned in Module 2.

## Rebuild the cleaned dataset

**Exercise:** Reuse the functions you completed in Module 2 to rebuild `enriched`: scope `train_df` to all four cities and `DEFAULT_COLUMNS`, drop columns above a 0.7 missing-value threshold, fill what remains per city, then add temporal features — the exact same sequence as `workflows.run_advanced`, minus the final split.

In [1]:
# TODO: rebuild enriched, reusing the Module 2 functions you already completed
from pathlib import Path
import sys

# Make the project's src/ package importable even if Jupyter starts in another folder/kernel.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "air_quality").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

import pandas as pd
from sklearn.linear_model import LinearRegression

from air_quality import data, evaluation, features
from air_quality.workflows import DEFAULT_COLUMNS

ALL_CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]

train_df, _ = data.load_datasets()
scoped = data.restrict_to_scope(train_df, ALL_CITIES, DEFAULT_COLUMNS)

columns_to_drop = data.columns_above_missing_threshold(scoped, threshold=0.7)
cleaned = data.drop_columns(scoped, columns_to_drop)

fillable_columns = [col for col in cleaned.columns if col not in ("city", "date")]
cleaned = data.fill_missing_by_city(cleaned, fillable_columns)

enriched = features.add_temporal_features(cleaned)
feature_cols = features.feature_columns(enriched)
enriched

,city,date,hour,site_latitude,site_longitude,pm2_5,sulphurdioxide_so2_column_number_density,carbonmonoxide_co_column_number_density,nitrogendioxide_no2_column_number_density,formaldehyde_tropospheric_hcho_column_number_density,uvaerosolindex_absorbing_aerosol_index,ozone_o3_column_number_density,cloud_cloud_fraction,month,dayofweek
7946,Bujumbura,2023-11-04,11,-3.380650,29.383830,27.504000,0.000053,0.033598,0.000053,0.000047,-0.685403,0.116598,0.827591,11,5
7947,Bujumbura,2023-11-08,12,-3.380650,29.383830,17.272750,0.000053,0.033598,0.000053,0.000047,0.064541,0.115969,0.827591,11,2
7948,Bujumbura,2023-11-09,11,-3.380650,29.383830,6.688182,0.000053,0.033598,0.000053,0.000047,-0.136088,0.115215,1.000000,11,3
7949,Bujumbura,2023-11-10,11,-3.380650,29.383830,15.051000,0.000053,0.033598,0.000053,0.000047,-0.222847,0.118663,1.000000,11,4
7950,Bujumbura,2023-11-15,11,-3.380650,29.383830,5.863500,0.000053,0.033598,0.000053,0.000047,-0.099786,0.114286,1.000000,11,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5813,Nairobi,2024-01-27,10,-1.283724,36.810364,13.334286,0.000199,0.029279,0.000069,0.000270,-0.970830,0.110373,0.047826,1,5
7284,Nairobi,2024-01-27,10,-1.286301,36.830761,21.272407,-0.000128,0.029279,0.000053,0.000158,-0.842223,0.109551,0.154551,1,5
7611,Nairobi,2024-01-27,10,-1.235120,36.812150,9.172387,0.000299,0.027913,0.000042,0.000287,-1.320420,0.109880,0.017744,1,5
7928,Nairobi,2024-01-27,10,-1.224292,36.805005,5.685071,0.000184,0.027913,0.000043,0.000265,-1.316598,0.109739,0.017744,1,5


## One pair only tells you about that pair

Recall Session 2's numbers:

| Direction | RMSE | R² |
|---|---|---|
| Train Kampala → test Nairobi | 24.6 | -0.02 |
| Train Nairobi → test Kampala | 15.2 | -0.16 |

Neither number tells you anything about Lagos or Bujumbura — they never appeared in either run. Would the model do better or worse there? You cannot know from a single manual pair, and picking a *different* pair would have given you yet another, equally partial answer.

[`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) automates the *idea* behind the manual split — never mix a city across train and validation — but applies it systematically: with one fold per city, each city takes a turn being the one the model has never seen, while training on the other three. This is sometimes called leave-one-group-out.

## Implement it

Open `src/air_quality/evaluation.py` and complete `evaluate_group_cv`, in the "Session 3 — optional modules" section at the bottom of the file. Its docstring specifies the exact contract, and `tests/test_evaluation_advanced.py` specifies the exact expected behavior — read it the same way you read `tests/test_evaluation.py` in Session 2. One detail the docstring calls out: fit a fresh [`clone()`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html) of the model for each fold, not the same instance refit repeatedly — a fold must never carry over anything learned by a previous fold. Run `uv run pytest tests/test_evaluation_advanced.py -v` until it passes, then come back here.

## Apply it, on all four cities at once

**Exercise:** Call `evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")`, then look at the per-fold breakdown as a table.

In [2]:
# Given: once evaluate_group_cv is implemented, this just calls it
result = evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")

pd.DataFrame(result["folds"]).set_index("test_group")

,rmse,mae,r2
test_group,,,
Kampala,13.402341,9.666581,0.127175
Nairobi,25.191590,10.150797,-0.017510
Lagos,52.008687,22.874298,0.030222
Bujumbura,19.736939,12.608173,0.214008


In [3]:
# Given: the aggregated view
{k: v for k, v in result.items() if k != "folds"}

{'rmse_mean': np.float64(27.584889349609792),
 'rmse_std': np.float64(14.705311521136801),
 'mae_mean': np.float64(13.82496240738647),
 'mae_std': np.float64(5.342346691969214),
 'r2_mean': np.float64(0.08847359915786224),
 'r2_std': np.float64(0.08927825005641958)}

**Look at the per-city RMSE.** One city is far harder to predict for than the other three — the kind of gap a single Kampala/Nairobi comparison could never have revealed, in either direction.

**Continue on the site:** [Module 3 — Robust Evaluation](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_3/)